In [1]:
import os
from pathlib import Path
import numpy as np
from tqdm import tqdm

import plotly.graph_objects as go
from Bio.PDB import PDBParser
import py3Dmol

from mdm2_breaker import ProteinFeaturizer

ROOT = Path(os.getcwd()).parents[0]

In [2]:
pdb_file = os.path.join(ROOT, "data", "MDM2_Breaker", "1YCR.pdb")

## Parse the Protein

In [ ]:
protein = ProteinFeaturizer(pdb_file = pdb_file)
coords, _ = protein._parse_structure()


In [4]:
def view_protein(pdb_file, highlight_chain="A"):
    view = py3Dmol.view(query=pdb_file)

    # Show the whole protein as a "Cartoon" (Ribbon)
    view.setStyle({"cartoon": {"color": "spectrum"}})

    # Show the Alpha Carbons as spheres
    view.addStyle(
        {"chain": highlight_chain, "atom": "CA"},
        {"sphere": {"radius": 0.5, "color": "red"}},
    )

    view.zoomTo()
    view.show()

view_protein(pdb_file)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
def plot_beads(coords, title="MDM2 Alpha Carbons"):
    x, y, z = coords.T

    x_plot, y_plot, z_plot = [], [], []
    for i in range(len(coords) - 1):
        # Add current point
        x_plot.append(x[i])
        y_plot.append(y[i])
        z_plot.append(z[i])
        
        # Check physics: Distance to next point
        dist = np.linalg.norm(coords[i] - coords[i+1])
        # Break the line if distance is more than the 3.8A peptide bond
        if dist > 4.0:             
            x_plot.append(None)
            y_plot.append(None)
            z_plot.append(None)
            
    # Add the very last point
    x_plot.append(x[-1])
    y_plot.append(y[-1])
    z_plot.append(z[-1])

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=x_plot,
                y=y_plot,
                z=z_plot,
                mode="markers+lines",  # Lines connect the sequence backbone
                marker=dict(size=5, color=z, colorscale="Viridis", opacity=0.8),
                line=dict(color="darkblue", width=2),
            )
        ]
    )

    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="X (Å)", yaxis_title="Y (Å)", zaxis_title="Z (Å)"),
        margin=dict(l=0, r=0, b=0, t=0),
    )

    fig.show()


plot_beads(coords)

## Parse the Small Molecules